# Uzbek alphabet amendment study
### What changing sh, ch, oʻ, gʻ → ş, ç, ö, ğ does to real Uzbek text and to tokenizers

Measured on the same 474-document deduplicated held-out set used in the paper.

| Step | Output |
|---|---|
| 1. Self-test the converter | 48 checks |
| 2. Rebuild the held-out set | `uz_heldout_dedup.jsonl`, checked against the paper's counts |
| 3. Check the published tokenizer's apostrophe handling | one line: built in or not |
| 4. Run the study | apostrophe census, conversion audit, tokenizer fertility on three versions of the text |

**Settings:** Accelerator **None** (CPU is enough). Internet **on**. Add Input: the **uzbek-fineweb2-tokens-16k** dataset. About 10–15 minutes.

In [ ]:
!pip install -q tiktoken sentencepiece protobuf

### Save the three scripts as files
Each cell below writes one `.py` file to disk. Run them; they print `Writing ...` and nothing else.

In [ ]:
%%writefile uzbek_imlo.py
#!/usr/bin/env python3
"""
uzbek_imlo.py - convert Uzbek Latin text to the amended alphabet

    sh -> ş    ch -> ç    oʻ -> ö    gʻ -> ğ        (capitals: Ş Ç Ö Ğ)

Mapping as reported for the 2026 amendment approved by parliament. Verify it against
the final signed text (lex.uz) before publishing results. The tutuq belgisi (ʼ) is
left unchanged by this script.

Three functions:

  naive_convert(text)              plain find-and-replace. The baseline.
  careful_convert(text, log=None)  leaves URLs, e-mails, @handles, #hashtags and
                                   foreign-looking words alone; accepts every apostrophe
                                   character people actually type; reads a word-final o'
                                   as a closing quote; counts every decision in `log`.
  normalize_apostrophes(text)      same alphabet, consistent marks:
                                   oʻ/gʻ -> U+02BB, in-word tutuq -> U+02BC.
  to_old(text)                     the reverse: ş ç ö ğ -> sh ch oʻ gʻ.

Command line:

  python uzbek_imlo.py --test
  python uzbek_imlo.py "Men o'qishni yaxshi ko'raman"        # Men öqişni yaxşi köraman
  python uzbek_imlo.py -i in.txt -o out.txt --mode normalize+careful

  Streams line by line, so it works on a multi-GB corpus. A ~200 MB sample is
  plenty for training a 16k-vocabulary tokenizer.

In a Jupyter / Kaggle notebook, save this file first (%%writefile uzbek_imlo.py),
then either run it with !python uzbek_imlo.py --test, or import it:

  from uzbek_imlo import careful_convert, naive_convert, normalize_apostrophes
  main(["--test"])            # the CLI also accepts an argument list

Known limits (the log counts each one, so you can measure them instead of guessing):
  * the bare letter name "o'" standing alone is read as a closing quote
  * English words built only from letters Uzbek also uses ("fresh", "such") are only
    skipped if they are in FOREIGN_WORDS - extend the list from the audit's top words
  * a word containing c not followed by h is treated as foreign, so Uzbek words spelled
    with a nonstandard c ("licenziyalash" for "litsenziyalash") are left unconverted
  * a foreign name glued to an Uzbek suffix by an apostrophe ("Samsungʻning") is
    converted as if it were Uzbek
"""

import argparse
import re
import sys
from collections import Counter

# Every character people type between o/g and the next letter, in the wild.
APOS_CHARS = "'\u2018\u2019\u02BB\u02BC\u0060\u00B4"   # ' ‘ ’ ʻ ʼ ` ´
OKINA = "\u02BB"   # ʻ  official mark in oʻ / gʻ
TUTUQ = "\u02BC"   # ʼ  official tutuq belgisi
NEW = {"s": "ş", "S": "Ş", "c": "ç", "C": "Ç", "o": "ö", "O": "Ö", "g": "ğ", "G": "Ğ"}

_AP = re.escape(APOS_CHARS)
_A = "[" + _AP + "]"
_L = r"[^\W\d_]"                     # any Unicode letter

_PROTECT_RE = re.compile(
    r"(?:https?://|www\.)\S+"                                          # URLs
    r"|[\w.+-]+@[\w-]+(?:\.[\w-]+)+"                                   # e-mails
    r"|(?<![\w" + _AP + r"])[@#][\w" + _AP + r"]+"                     # @handles, #hashtags
    r"|\b[\w-]+(?:\.[\w-]+)*\.(?:uz|com|ru|org|net|io|info|me|tv|gov|edu)\b(?:/\S*)?",
    re.IGNORECASE,
)
_WORD_RE = re.compile("(?:" + _L + "|" + _A + ")+")
_TARGET_RE = re.compile("[sScC][hH]|[oOgG]" + _A)
_HAS_APOS_RE = re.compile(_A)

# Letters or clusters native Uzbek Latin never uses: no w, no c outside "ch", etc.
# ("sch" and "tch" are deliberately absent: ruscha, forscha, nemischa are Uzbek.)
_FOREIGN_RE = re.compile(r"w|c(?!h)|th|ph|ck|ght", re.IGNORECASE)

# Foreign words the letter rules cannot catch. Extend from 2_conversion_audit.md.
FOREIGN_WORDS = {
    "english", "british", "spanish", "fresh", "flash", "cash", "push", "bash", "brush",
    "finish", "such", "much", "each", "rich", "teach", "reach", "beach", "match",
    "school", "machine", "dashboard", "hashtag", "share", "shop", "shopping", "shorts",
    "shell", "ssh", "sha", "chmod", "tech", "technology", "technical", "chrome",
    "chatgpt", "chevrolet", "chery", "changan", "chelsea", "michael", "charles",
    "chris", "christian", "christmas",
}


def _og_decision(word, i, quoted):
    """Is o/g + apostrophe at word[i] the letter oʻ/gʻ, or a quote mark?"""
    ch, n = word[i], len(word)
    prev = word[i - 1] if i > 0 else ""
    after = word[i + 2] if i + 2 < n else ""
    after2 = word[i + 3] if i + 3 < n else ""
    if ch == "O" and after.isupper() and after2.islower():
        return "skip_foreign_name"             # O'Reilly, O'Neil
    if after.isalpha() or (after and after in APOS_CHARS and after2.isalpha()):
        return "convert_o_g"                   # oʻzbek, moʻʼjiza, qoʻngʻiroq, boshlangʻich
    if ch in "gG" and prev in ("n", "N"):
        return "skip_ng_quote"                 # word-final: 'rang' - the mark closes a quote
    if ch in "gG" and not quoted:
        return "convert_o_g"                   # word-final gʻ: togʻ, bogʻ, sogʻ
    return "skip_final_quote"                  # 'dunyo' - the mark closes a quote


def _process_word(word, log, mode):
    """mode 'convert' = new letters; mode 'normalize' = same alphabet, canonical marks."""
    has_target = _TARGET_RE.search(word) is not None
    if mode == "convert" and not has_target:
        return word
    if mode == "normalize" and not _HAS_APOS_RE.search(word):
        return word
    if has_target:
        core = word.strip(APOS_CHARS)
        if core.lower() in FOREIGN_WORDS or _FOREIGN_RE.search(core):
            log["skip_foreign_word"] += 1
            return word

    out, i, n = [], 0, len(word)
    quoted = word[0] in APOS_CHARS
    while i < n:
        ch = word[i]
        nxt = word[i + 1] if i + 1 < n else ""
        if ch in "oOgG" and nxt and nxt in APOS_CHARS:
            decision = _og_decision(word, i, quoted)
            log[decision] += 1
            if decision == "convert_o_g":
                out.append(ch + OKINA if mode == "normalize" else NEW[ch])
                i += 2
                continue
        elif mode == "convert" and ch in "sScC" and nxt in ("h", "H"):
            log["convert_sh_ch"] += 1
            out.append(NEW[ch])
            i += 2
            continue
        elif mode == "normalize" and ch in APOS_CHARS and out and out[-1].isalpha() and nxt.isalpha():
            out.append(TUTUQ)                  # maʼlumot, moʻʼjiza
            i += 1
            continue
        out.append(ch)
        i += 1
    return "".join(out)


def _apply(text, fn, log):
    """Run fn on everything except protected spans (URLs, e-mails, handles, domains)."""
    out, pos = [], 0
    for m in _PROTECT_RE.finditer(text):
        out.append(fn(text[pos:m.start()]))
        if _TARGET_RE.search(m.group(0)):
            log["skip_protected"] += 1
        out.append(m.group(0))
        pos = m.end()
    out.append(fn(text[pos:]))
    return "".join(out)


def careful_convert(text, log=None):
    log = Counter() if log is None else log
    return _apply(text, lambda s: _WORD_RE.sub(lambda m: _process_word(m.group(0), log, "convert"), s), log)


def normalize_apostrophes(text, log=None):
    log = Counter() if log is None else log
    return _apply(text, lambda s: _WORD_RE.sub(lambda m: _process_word(m.group(0), log, "normalize"), s), log)


_TO_OLD = {"ö": "o" + OKINA, "Ö": "O" + OKINA, "ğ": "g" + OKINA, "Ğ": "G" + OKINA}


def to_old(text):
    """Amended alphabet -> current alphabet: ş->sh, ç->ch, ö->oʻ, ğ->gʻ.

    Lets a tokenizer and model trained on the current alphabet read new-alphabet text
    with no retraining. Capital Ş/Ç become SH/CH inside an all-caps word, Sh/Ch otherwise.
    """
    out, n = [], len(text)
    for i, ch in enumerate(text):
        if ch in "şç":
            out.append(ch.replace("ş", "sh").replace("ç", "ch"))
        elif ch in "ŞÇ":
            nxt = text[i + 1] if i + 1 < n else ""
            h = "H" if nxt.isalpha() and nxt.isupper() else "h"
            out.append(("S" if ch == "Ş" else "C") + h)
        else:
            out.append(_TO_OLD.get(ch, ch))
    return "".join(out)


_NAIVE = [(b + a, NEW[b]) for b in "oOgG" for a in APOS_CHARS] + [
    ("sh", "ş"), ("Sh", "Ş"), ("SH", "Ş"), ("sH", "ş"),
    ("ch", "ç"), ("Ch", "Ç"), ("CH", "Ç"), ("cH", "ç"),
]


def naive_convert(text):
    for old, new in _NAIVE:
        text = text.replace(old, new)
    return text


# ---------------------------------------------------------------------------- tests
_SAME = object()
_CASES = [  # input, careful, naive (_SAME = identical to careful; None = input unchanged)
    ("Men o'qishni yaxshi ko'raman", "Men öqişni yaxşi köraman", _SAME),
    ("O‘zbekiston, oʻzbek, o’zbek, o`zbek, O'ZBEKISTON", "Özbekiston, özbek, özbek, özbek, ÖZBEKISTON", _SAME),
    ("G'ijduvon, tog' va bog'", "Ğijduvon, toğ va boğ", _SAME),
    ("SHAHAR, Shahar, shahar", "ŞAHAR, Şahar, şahar", _SAME),
    ("uchun, chiroyli, MChJ, ShHT", "uçun, çiroyli, MÇJ, ŞHT", _SAME),
    ("ruscha, forscha", "rusça, forsça", _SAME),
    ("mo''jiza, ma'lumot, Is'hoq", "mö'jiza, ma'lumot, Is'hoq", _SAME),
    ("https://kun.uz/news/shahar-yangiliklari", None, "https://kun.uz/news/şahar-yangiliklari"),
    ("shahnoza@mail.uz", None, "şahnoza@mail.uz"),
    ("@shahar_uz #O'zbekiston", None, "@şahar_uz #Özbekiston"),
    ("ChatGPT, Chevrolet, English", None, "ÇatGPT, Çevrolet, Engliş"),
    ("U 'dunyo' dedi", None, "U 'dunyö dedi"),
    ("O'Reilly", None, "ÖReilly"),
    ("'rang'", None, "'ranğ"),
    ("qo'ng'iroq, boshlangʻich, yong‘in", "qönğiroq, boşlanğiç, yonğin", _SAME),   # ngʻ inside a word
    ("Qorongʻi va shoʻngʻin", "Qoronği va şönğin", _SAME),
]
_NORMALIZE_CASES = [
    ("o'zbek ma'lumot", "oʻzbek maʼlumot"),
    ("mo''jiza", "moʻʼjiza"),
    ("tog'", "togʻ"),
    ("'dunyo'", "'dunyo'"),
    ("oʻzbek maʼlumot", "oʻzbek maʼlumot"),          # already canonical: unchanged
]


def _selftest():
    failures = 0

    def check(label, got, want):
        nonlocal failures
        ok = got == want
        failures += not ok
        print(("ok    " if ok else "FAIL  ") + label + ("" if ok else f"\n      got:  {got!r}\n      want: {want!r}"))

    for text, careful, naive in _CASES:
        careful = text if careful is None else careful
        naive = careful if naive is _SAME else naive
        check(f"careful  {text}", careful_convert(text), careful)
        check(f"naive    {text}", naive_convert(text), naive)
        check(f"words    {text}", len(careful_convert(text).split()), len(text.split()))
    for text, want in _NORMALIZE_CASES:
        check(f"normal.  {text}", normalize_apostrophes(text), want)
    check("pipeline normalize+careful", careful_convert(normalize_apostrophes("Men o'qishni yaxshi ko'raman")),
          "Men öqişni yaxşi köraman")
    for text in ["Men o'qishni yaxshi ko'raman", "G'ijduvon, tog' va bog'", "SHAHAR va Choʻl", "qoʻngʻiroq chalindi"]:
        norm = normalize_apostrophes(text)
        check(f"round-trip  {text}", to_old(careful_convert(norm)), norm)
    print(f"\n{failures} failure(s)")
    return 1 if failures else 0


def main(argv=None):
    """Command-line entry point. Pass argv as a list to call it from a notebook."""
    ap = argparse.ArgumentParser(description="Uzbek Latin -> amended alphabet (sh ch oʻ gʻ -> ş ç ö ğ)")
    ap.add_argument("text", nargs="?", help="a string to convert")
    ap.add_argument("-i", "--input", help="input file (default: stdin)")
    ap.add_argument("-o", "--output", help="output file (default: stdout)")
    ap.add_argument("--mode", default="careful", choices=["careful", "naive", "normalize", "normalize+careful"])
    ap.add_argument("--test", action="store_true", help="run the self-tests")
    a = ap.parse_args(argv)
    if a.test:
        return _selftest()

    log = Counter()
    fn = {
        "careful": lambda s: careful_convert(s, log),
        "naive": naive_convert,
        "normalize": lambda s: normalize_apostrophes(s, log),
        "normalize+careful": lambda s: careful_convert(normalize_apostrophes(s), log),
    }[a.mode]

    if a.text is not None:
        print(fn(a.text))
        return 0
    src = open(a.input, encoding="utf-8") if a.input else sys.stdin
    dst = open(a.output, "w", encoding="utf-8") if a.output else sys.stdout
    for line in src:
        dst.write(fn(line))
    if src is not sys.stdin:
        src.close()
    if dst is not sys.stdout:
        dst.close()
    if log:
        print("decisions: " + ", ".join(f"{k}={v:,}" for k, v in log.most_common()), file=sys.stderr)
    return 0


def _in_notebook():
    """True inside Jupyter/Kaggle, where sys.argv belongs to the kernel, not to us."""
    try:
        get_ipython  # noqa: F821  (defined by IPython)
        return True
    except NameError:
        return False


if __name__ == "__main__" and not _in_notebook():
    sys.exit(main())


In [ ]:
%%writefile alphabet_study.py
#!/usr/bin/env python3
"""
alphabet_study.py - what Uzbekistan's alphabet amendment (sh ch oʻ gʻ -> ş ç ö ğ) does
to real Uzbek text and to tokenizers. CPU only; a few minutes on the 474-document
deduplicated held-out set from the paper.

  pip install tiktoken transformers tokenizers sentencepiece protobuf
  python build_heldout.py                        # rebuilds the 474 documents, one per line
  python alphabet_study.py --heldout uz_heldout_dedup.jsonl --out results/
  python alphabet_study.py --heldout uz_heldout_dedup.jsonl --skip-tokenizers   # audit only

  Use the .jsonl file: it keeps each of the 474 documents separate. The paper's
  uz_heldout_dedup.txt joins documents with single newlines, so their boundaries are lost.

In a Jupyter / Kaggle notebook: save uzbek_imlo.py and this file with %%writefile, then run
  !python alphabet_study.py --heldout ... --out ...
or call main(["--heldout", "...", "--out", "..."]) after importing.

Writes to --out:

  1_apostrophes.md       which characters people actually type in oʻ / gʻ
  2_conversion_audit.md  how much text changes, and where plain find-and-replace and the
                         careful converter disagree (top words first)
  2_disagreements.csv    random sample of disagreements. Fill in the `verdict` column by
                         hand: naive_wrong / careful_wrong / both_ok / unclear. That turns
                         "naive replacement breaks things" into a measured error rate.
  3_tokenizers.md/.csv   fertility and bytes/token for every tokenizer on three versions
                         of the same text:
                           raw  = the web text as it is
                           norm = same alphabet, apostrophes made consistent
                           new  = the amended alphabet (careful converter applied to norm)
                         raw -> norm isolates the apostrophe mess; norm -> new isolates the
                         new letters.

Retraining (this is the part that tests "the tokenizer is the cheap lever"):
  1. Take a ~200 MB sample of training text (decode train.bin, or re-stream FineWeb-2).
  2. python uzbek_imlo.py -i sample.txt -o sample_norm.txt --mode normalize
     python uzbek_imlo.py -i sample.txt -o sample_new.txt  --mode normalize+careful
  3. With YOUR training script and identical settings (16,384 vocab), train one tokenizer
     on sample_norm.txt and one on sample_new.txt. Train both on the same sample - do not
     compare a new-alphabet tokenizer against the original uzbek-bpe-16k, which saw far
     more text, or data size becomes a confound.
  4. Re-run with --extra "retrained-old=file:tok_norm.json" "retrained-new=file:tok_new.json"

Notes:
  * Word counts are identical in all three versions (conversion never touches
    whitespace), so a fertility difference is purely a token-count difference.
  * Byte counts are NOT identical (oʻ is 2-4 bytes, ö is 2). If you go on to score
    models on converted text, don't compare bits-per-byte across spellings; use bits
    per word.
  * Make TOKENIZERS match the exact IDs of your paper's Table 1 benchmark.
"""

import argparse
import csv
import json
import os
import random
from collections import Counter

try:
    from uzbek_imlo import APOS_CHARS, careful_convert, naive_convert, normalize_apostrophes, to_old
except ModuleNotFoundError:
    # Pasted into a notebook cell after uzbek_imlo.py's code was run in an earlier cell:
    # the functions already exist in the notebook's namespace.
    _need = ("APOS_CHARS", "careful_convert", "naive_convert", "normalize_apostrophes", "to_old")
    if not all(n in globals() for n in _need):
        raise ModuleNotFoundError(
            "uzbek_imlo.py not found. Save it next to this script first "
            "(in a notebook: put %%writefile uzbek_imlo.py at the top of its cell and run it).")

TOKENIZERS = [  # (label, spec)  spec = hf:<repo> | tiktoken:<encoding> | file:<tokenizer.json>
                #                  prefix rev: = convert ş ç ö ğ back to sh ch oʻ gʻ before tokenizing
    ("uzbek-bpe-16k", "hf:IslombekT/uzbek-bpe-16k"),
    ("uzbek-bpe-16k + new→old map", "rev:hf:IslombekT/uzbek-bpe-16k"),
    ("XLM-RoBERTa", "hf:FacebookAI/xlm-roberta-base"),
    ("XGLM", "hf:facebook/xglm-564M"),
    ("mT5", "hf:google/mt5-base"),
    ("GPT-4o o200k_base", "tiktoken:o200k_base"),
    ("mBERT", "hf:google-bert/bert-base-multilingual-cased"),
    ("BLOOM", "hf:bigscience/bloom-560m"),
    ("Qwen2.5", "hf:Qwen/Qwen2.5-0.5B"),
    ("GPT-3.5/4 cl100k_base", "tiktoken:cl100k_base"),
    ("GPT-2", "tiktoken:gpt2"),
    ("mGPT", "hf:ai-forever/mGPT"),
]

APOS_LABELS = {
    "'": "U+0027 ' (ASCII apostrophe)",
    "\u2018": "U+2018 ‘ (left single quote)",
    "\u2019": "U+2019 ’ (right single quote)",
    "\u02BB": "U+02BB ʻ (official oʻ/gʻ mark)",
    "\u02BC": "U+02BC ʼ (official tutuq mark)",
    "`": "U+0060 ` (backtick)",
    "\u00B4": "U+00B4 ´ (acute accent)",
}


DEFAULT_HELDOUT = ["uz_heldout_dedup.jsonl", "/kaggle/working/uz_heldout_dedup.jsonl",
                   "uz_heldout_dedup.txt", "/kaggle/working/uz_heldout_dedup.txt"]


def find_heldout(path):
    if path:
        if not os.path.exists(path):
            raise FileNotFoundError(f"{path} not found")
        return path
    for p in DEFAULT_HELDOUT:
        if os.path.exists(p):
            return p
    raise FileNotFoundError("no held-out file found. Run build_heldout.py first, or pass --heldout.")


def load_passages(path, sep):
    if path.endswith(".jsonl"):
        with open(path, encoding="utf-8") as f:
            return [json.loads(line)["text"] for line in f if line.strip()]
    with open(path, encoding="utf-8") as f:
        return [p for p in f.read().split(sep) if p.strip()]


def apostrophe_census(passages):
    counts = Counter()
    for p in passages:
        for a, b in zip(p, p[1:]):
            if a in "oOgG" and b in APOS_CHARS:
                counts[b] += 1
    return counts


def conversion_audit(passages, seed):
    log, top, example = Counter(), Counter(), {}
    words = changed = passages_changed = passages_disagree = 0
    rows = []
    for pid, raw in enumerate(passages):
        careful = careful_convert(raw, log)
        naive = naive_convert(raw)
        r, c, n = raw.split(), careful.split(), naive.split()
        assert len(r) == len(c) == len(n), "conversion changed whitespace"
        words += len(r)
        any_changed = False
        for j, (rw, cw, nw) in enumerate(zip(r, c, n)):
            if cw != rw:
                changed += 1
                any_changed = True
            if cw != nw:
                top[rw] += 1
                example.setdefault(rw, (nw, cw))
                rows.append({"passage": pid, "original": rw, "naive": nw, "careful": cw,
                             "context": " ".join(r[max(0, j - 6): j + 7]), "verdict": ""})
        passages_changed += any_changed
        passages_disagree += careful != naive
    random.Random(seed).shuffle(rows)
    return dict(log=log, top=top, example=example, rows=rows, words=words, changed=changed,
                passages=len(passages), passages_changed=passages_changed,
                passages_disagree=passages_disagree)


def token_counter(spec):
    if spec.startswith("rev:"):
        inner = token_counter(spec[4:])
        return lambda s: inner(to_old(s))
    kind, name = spec.split(":", 1)
    if kind == "tiktoken":
        import tiktoken
        enc = tiktoken.get_encoding(name)
        return lambda s: len(enc.encode_ordinary(s))
    if kind == "file":
        from tokenizers import Tokenizer
        tok = Tokenizer.from_file(name)
        return lambda s: len(tok.encode(s, add_special_tokens=False).ids)
    if kind == "hf":
        try:
            from transformers import AutoTokenizer
            from transformers.utils import logging as hf_logging
            hf_logging.set_verbosity_error()        # silence "sequence too long" warnings
            tok = AutoTokenizer.from_pretrained(name)
            return lambda s: len(tok.encode(s, add_special_tokens=False))
        except Exception:
            from tokenizers import Tokenizer
            tok = Tokenizer.from_pretrained(name)
            return lambda s: len(tok.encode(s, add_special_tokens=False).ids)
    raise ValueError(f"unknown tokenizer spec: {spec}")


def pct(new, old):
    return f"{(new - old) / old * 100:+.1f}%"


def write_census(path, census):
    total = sum(census.values()) or 1
    lines = ["# Which mark do people type in oʻ / gʻ?", "",
             "Counts every o/O/g/G followed by an apostrophe-like character.", "",
             "| Character | Count | Share |", "|---|---:|---:|"]
    for ch, k in census.most_common():
        lines.append(f"| {APOS_LABELS.get(ch, repr(ch))} | {k:,} | {k / total:.1%} |")
    if census and not (census.get("'") or census.get("\u2019") or census.get("`")):
        lines += ["", "**Warning:** no ASCII apostrophes, right quotes or backticks at all. Real web "
                  "text always has them, so this text was probably normalized before it was saved "
                  "(for example, decoded from a tokenizer that maps them to ʻ). This table then shows "
                  "which marks survived normalization, not what people type."]
    open(path, "w", encoding="utf-8").write("\n".join(lines) + "\n")


def write_audit(path_md, path_csv, au, n_sample):
    w = au["words"] or 1
    lines = [
        "# Conversion audit", "",
        f"- Passages: {au['passages']:,} · words: {au['words']:,}",
        f"- Words whose spelling changes (careful converter): {au['changed']:,} ({au['changed'] / w:.1%})",
        f"- Passages with at least one changed word: {au['passages_changed']:,} "
        f"({au['passages_changed'] / max(au['passages'], 1):.1%})",
        f"- Passages where naive find-and-replace and the careful converter disagree: "
        f"{au['passages_disagree']:,} ({au['passages_disagree'] / max(au['passages'], 1):.1%})",
        f"- Word-level disagreements: {len(au['rows']):,}", "",
        "## Careful converter decisions", "", "| Decision | Count |", "|---|---:|",
    ]
    lines += [f"| {k} | {v:,} |" for k, v in au["log"].most_common()]
    lines += ["", "## Top 40 disagreements", "", "| Original | Times | Naive | Careful |", "|---|---:|---|---|"]
    for word, k in au["top"].most_common(40):
        nw, cw = au["example"][word]
        lines.append(f"| `{word}` | {k:,} | `{nw}` | `{cw}` |")
    lines += ["", f"A random sample of {min(n_sample, len(au['rows']))} disagreements is in "
              f"`{os.path.basename(path_csv)}` — label the `verdict` column."]
    open(path_md, "w", encoding="utf-8").write("\n".join(lines) + "\n")
    with open(path_csv, "w", encoding="utf-8", newline="") as f:
        wr = csv.DictWriter(f, fieldnames=["passage", "original", "naive", "careful", "context", "verdict"])
        wr.writeheader()
        wr.writerows(au["rows"][:n_sample])


def tokenizer_table(passages, specs):
    versions = {"raw": passages}
    versions["norm"] = [normalize_apostrophes(p) for p in passages]
    versions["new"] = [careful_convert(p) for p in versions["norm"]]
    words = sum(len(p.split()) for p in passages)
    nbytes = {k: sum(len(p.encode("utf-8")) for p in v) for k, v in versions.items()}
    rows = []
    for label, spec in specs:
        try:
            count = token_counter(spec)
        except Exception as e:                      # keep going if one tokenizer fails
            print(f"  skipped {label}: {e}")
            continue
        toks = {k: sum(count(p) for p in v) for k, v in versions.items()}
        rows.append({"tokenizer": label,
                     **{f"fertility_{k}": toks[k] / words for k in versions},
                     **{f"bytes_per_token_{k}": nbytes[k] / toks[k] for k in versions}})
        print(f"  {label:<24} raw {toks['raw'] / words:.3f}   norm {toks['norm'] / words:.3f}   "
              f"new {toks['new'] / words:.3f}")
    return rows, words, nbytes


def write_tokenizers(path_md, path_csv, rows, words, nbytes):
    rows = sorted(rows, key=lambda r: r["fertility_new"])
    lines = ["# Tokenizers: current vs. amended alphabet", "",
             "Fertility = tokens per word (lower is better). Same words in every column.", "",
             "| Tokenizer | Raw | Norm | New | Raw→New | Norm→New | Bytes/token (new) |",
             "|---|---:|---:|---:|---:|---:|---:|"]
    for r in rows:
        lines.append(f"| {r['tokenizer']} | {r['fertility_raw']:.3f} | {r['fertility_norm']:.3f} | "
                     f"{r['fertility_new']:.3f} | {pct(r['fertility_new'], r['fertility_raw'])} | "
                     f"{pct(r['fertility_new'], r['fertility_norm'])} | {r['bytes_per_token_new']:.3f} |")
    lines += ["", f"Words: {words:,}. Bytes: raw {nbytes['raw']:,} · norm {nbytes['norm']:,} · "
              f"new {nbytes['new']:,}.", "",
              "raw = web text as-is · norm = same alphabet, oʻ/gʻ on U+02BB and tutuq on U+02BC · "
              "new = norm converted to Ş Ç Ö Ğ.",
              "If a tokenizer applies its own apostrophe normalization before encoding, its "
              "published fertility corresponds to the Norm column.",
              "`+ new→old map` converts ş ç ö ğ back to sh ch oʻ gʻ before tokenizing: the "
              "zero-retraining way to keep a current-alphabet tokenizer and model working."]
    open(path_md, "w", encoding="utf-8").write("\n".join(lines) + "\n")
    with open(path_csv, "w", encoding="utf-8", newline="") as f:
        wr = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        wr.writeheader()
        wr.writerows(rows)


def parse_specs(items):
    return [tuple(s.split("=", 1)) for s in items]


def main(argv=None):
    ap = argparse.ArgumentParser(
        description="What the Uzbek alphabet amendment does to real text and to tokenizers.")
    ap.add_argument("--heldout", help=".jsonl with a 'text' field (preferred) or .txt split by --sep; "
                                      "default: uz_heldout_dedup.jsonl if present")
    ap.add_argument("--sep", default="\\n\\n", help="passage separator for .txt files (default: blank line)")
    ap.add_argument("--out", default="results")
    ap.add_argument("--sample", type=int, default=200, help="disagreements to export for hand-labelling")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--skip-tokenizers", action="store_true", help="audit only; no downloads")
    ap.add_argument("--only", nargs="*", help="replace the tokenizer list: label=spec ...")
    ap.add_argument("--extra", nargs="*", default=[], help="add tokenizers: label=spec ...")
    a = ap.parse_args(argv)

    os.makedirs(a.out, exist_ok=True)
    heldout = find_heldout(a.heldout)
    if heldout.endswith(".txt"):
        print("note: .txt input has no document boundaries; passages are split by --sep. "
              "Prefer the .jsonl from build_heldout.py.")
    passages = load_passages(heldout, a.sep.encode().decode("unicode_escape"))
    print(f"{len(passages):,} passages loaded from {heldout}")

    census = apostrophe_census(passages)
    write_census(os.path.join(a.out, "1_apostrophes.md"), census)
    total = sum(census.values()) or 1
    print("oʻ/gʻ marks: " + ", ".join(f"{APOS_LABELS.get(c, repr(c)).split(' (')[0]} {k / total:.0%}"
                                      for c, k in census.most_common()))
    if census and not (census.get("'") or census.get("\u2019") or census.get("`")):
        print("  warning: no ASCII/right-quote/backtick marks - this text looks pre-normalized")

    au = conversion_audit(passages, a.seed)
    write_audit(os.path.join(a.out, "2_conversion_audit.md"), os.path.join(a.out, "2_disagreements.csv"),
                au, a.sample)
    print(f"words changed by the amendment: {au['changed'] / max(au['words'], 1):.1%}; "
          f"naive and careful disagree in {au['passages_disagree']:,} of {au['passages']:,} passages")

    if not a.skip_tokenizers:
        specs = parse_specs(a.only) if a.only else TOKENIZERS
        specs += parse_specs(a.extra)
        print("tokenizers (fertility):")
        rows, words, nbytes = tokenizer_table(passages, specs)
        if rows:
            write_tokenizers(os.path.join(a.out, "3_tokenizers.md"), os.path.join(a.out, "3_tokenizers.csv"),
                             rows, words, nbytes)
    print(f"done -> {a.out}/")


def _in_notebook():
    """True inside Jupyter/Kaggle, where sys.argv belongs to the kernel, not to us."""
    try:
        get_ipython  # noqa: F821  (defined by IPython)
        return True
    except NameError:
        return False


if __name__ == "__main__" and not _in_notebook():
    main()


In [ ]:
%%writefile build_heldout.py
#!/usr/bin/env python3
"""
build_heldout.py - rebuild the paper's deduplicated held-out set in a fresh session.

Same algorithm as stages 1-2 of 02_clean_eval_and_score.py:
  1. fingerprint every 24-token window of train.bin (1 in 32 kept)
  2. walk val.bin document by document; keep only documents with ZERO fingerprint
     overlap, until 200,000 words

CPU only. About 3-5 minutes and ~4 GB of RAM on Kaggle. No GPU needed.

Writes to --out (default /kaggle/working, or the current folder outside Kaggle):
  uz_heldout_dedup.jsonl   one document per line, {"text": ...}   <- use this one
  uz_heldout_dedup.txt     documents joined by "\\n", byte-identical to the paper's file

Then checks the result against the paper: 474 documents kept, 306 rejected,
200,818 words, 1,646,167 bytes. If any number differs, the files are saved as
uz_heldout_dedup.MISMATCH.* instead, so the study cannot use them by accident.

Needs the Kaggle dataset uzbek-fineweb2-tokens-16k (train.bin, val.bin) as an input,
or --train / --val paths.
"""

import argparse
import glob
import json
import os
import sys
import time

import numpy as np

EOT_ID = 0
KGRAM = 24
KEEP_MASK = 31
MAX_DUP_FRAC = 0.0
MIN_DOC_WORDS = 30          # compared against the token count, exactly as in the paper's script
EVAL_WORDS = 200_000
TOK_REPO = "IslombekT/uzbek-bpe-16k"
EXPECTED = {"kept": 474, "rejected": 306, "words": 200_818, "bytes": 1_646_167}

_B = np.uint64(1000003)


def kgram_hashes(arr, K=KGRAM, mask=KEEP_MASK, chunk=20_000_000):
    """Rolling polynomial hash of every K-gram, subsampled to 1/(mask+1)."""
    out, n = [], len(arr)
    if n < K:
        return np.zeros(0, dtype=np.uint64)
    for s in range(0, n - K + 1, chunk):
        e = min(s + chunk, n - K + 1)
        m = e - s
        h = np.zeros(m, dtype=np.uint64)
        for j in range(K):
            h = h * _B + arr[s + j: s + j + m].astype(np.uint64)
        out.append(h[(h & np.uint64(mask)) == 0])
    return np.concatenate(out) if out else np.zeros(0, dtype=np.uint64)


def build(train_ids, val_ids, decode, verbose=True):
    """Return (kept_documents, n_rejected). `decode` maps a list of token ids to text."""
    t0 = time.time()
    idx = np.unique(kgram_hashes(np.asarray(train_ids)))
    if verbose:
        print(f"fingerprints: {len(idx):,} in {time.time() - t0:.0f}s")

    def dup_fraction(tok_ids):
        h = kgram_hashes(np.asarray(tok_ids, dtype=np.uint16))
        if len(h) == 0:
            return 0.0, 0
        p = np.searchsorted(idx, h)
        p[p >= len(idx)] = 0
        return float((idx[p] == h).mean()), len(h)

    kept, rejected, words, cur, i = [], 0, 0, [], 0
    while i < len(val_ids) and val_ids[i] != EOT_ID:     # start after a document boundary
        i += 1
    i += 1
    while i < len(val_ids) and words < EVAL_WORDS:
        t = int(val_ids[i])
        if t == EOT_ID:
            if len(cur) >= MIN_DOC_WORDS:
                frac, n_fp = dup_fraction(cur)
                if n_fp > 0 and frac <= MAX_DUP_FRAC:
                    d = decode(cur).strip()
                    if d:
                        kept.append(d)
                        words += len(d.split())
                else:
                    rejected += 1
            cur = []
        else:
            cur.append(t)
        i += 1
    return kept, rejected


def find_bins(train, val):
    if train and val:
        return train, val
    hits = glob.glob("/kaggle/input/**/train.bin", recursive=True) or glob.glob("**/train.bin", recursive=True)
    if not hits:
        raise FileNotFoundError("train.bin not found. On Kaggle: Add Input -> uzbek-fineweb2-tokens-16k. "
                                "Elsewhere: pass --train and --val.")
    return hits[0], os.path.join(os.path.dirname(hits[0]), "val.bin")


def main(argv=None):
    ap = argparse.ArgumentParser(description="Rebuild the paper's deduplicated held-out set.")
    ap.add_argument("--train", help="path to train.bin (default: search /kaggle/input)")
    ap.add_argument("--val", help="path to val.bin (default: next to train.bin)")
    ap.add_argument("--out", default="/kaggle/working" if os.path.isdir("/kaggle/working") else ".")
    a = ap.parse_args(argv)

    jsonl = os.path.join(a.out, "uz_heldout_dedup.jsonl")
    if os.path.exists(jsonl):
        n = sum(1 for _ in open(jsonl, encoding="utf-8"))
        print(f"{jsonl} already exists ({n} documents) - delete it to rebuild")
        return 0

    train_path, val_path = find_bins(a.train, a.val)
    train_ids = np.memmap(train_path, dtype=np.uint16, mode="r")
    val_ids = np.memmap(val_path, dtype=np.uint16, mode="r")
    print(f"train.bin {len(train_ids):,} tokens | val.bin {len(val_ids):,} tokens")

    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(TOK_REPO)
    tok.model_max_length = 10 ** 9

    kept, rejected = build(train_ids, val_ids, tok.decode)

    text = "\n".join(kept)
    got = {"kept": len(kept), "rejected": rejected, "words": len(text.split()),
           "bytes": len(text.encode("utf-8"))}
    print(f"kept {got['kept']} documents, rejected {got['rejected']} | "
          f"{got['words']:,} words | {got['bytes']:,} bytes")
    bad = {k: (got[k], EXPECTED[k]) for k in EXPECTED if got[k] != EXPECTED[k]}

    # On a mismatch, save under a different name so alphabet_study.py cannot pick it up.
    suffix = ".MISMATCH" if bad else ""
    os.makedirs(a.out, exist_ok=True)
    out_jsonl = os.path.join(a.out, f"uz_heldout_dedup{suffix}.jsonl")
    with open(out_jsonl, "w", encoding="utf-8") as f:
        for d in kept:
            f.write(json.dumps({"text": d}, ensure_ascii=False) + "\n")
    with open(os.path.join(a.out, f"uz_heldout_dedup{suffix}.txt"), "w", encoding="utf-8") as f:
        f.write(text)

    if bad:
        print("MISMATCH with the paper's held-out set (got, expected): " + str(bad))
        print(f"Saved as {out_jsonl} so the study will not use it by mistake.")
        return 1
    print("matches the paper's held-out set exactly")
    print(f"-> {jsonl}")
    return 0


def _in_notebook():
    try:
        get_ipython  # noqa: F821
        return True
    except NameError:
        return False


if __name__ == "__main__" and not _in_notebook():
    sys.exit(main())


## 1. Self-test the converter
Must end with `0 failure(s)`.

In [ ]:
!python uzbek_imlo.py --test | tail -5
!python uzbek_imlo.py "Men o'qishni yaxshi ko'raman. G'ijduvon, tog' va bog'."

## 2. Rebuild the paper's held-out set
Fingerprints every 24-token window of `train.bin`, then keeps only `val.bin` documents that share none of them. Must end with **matches the paper's held-out set exactly** (474 documents, 200,818 words, 1,646,167 bytes). If it doesn't, the files are saved under a `.MISMATCH` name and step 4 will refuse to run.

In [ ]:
!python build_heldout.py

## 3. Which apostrophe spellings does the published tokenizer normalize?
Each spelling of *oʻzbek* that gives the same tokens as the official `ʻ` is normalized inside the tokenizer. The last line prints the normalizer itself, so the model card can describe exactly what it does.

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("IslombekT/uzbek-bpe-16k")
variants = {
    "U+02BB ʻ (official)":    "o\u02BBzbek",
    "U+0027 ' (ASCII)":       "o'zbek",
    "U+2019 ’ (right quote)": "o\u2019zbek",
    "U+2018 ‘ (left quote)":  "o\u2018zbek",
    "U+0060 ` (backtick)":    "o`zbek",
}
ref = tok(variants["U+02BB ʻ (official)"], add_special_tokens=False)["input_ids"]
for label, text in variants.items():
    ids = tok(text, add_special_tokens=False)["input_ids"]
    print(f"{label:24} {str(ids):18} {'same as ʻ' if ids == ref else 'DIFFERENT - not normalized'}")
print("\nnormalizer:", tok.backend_tokenizer.normalizer)

## 4. Run the study
`raw` = the web text as it is · `norm` = same alphabet, apostrophes made consistent · `new` = the amended alphabet.
`raw → norm` isolates the apostrophe mess; `norm → new` isolates the new letters. Skipped tokenizers are reported and do not stop the run.

In [ ]:
!python alphabet_study.py --heldout /kaggle/working/uz_heldout_dedup.jsonl --out /kaggle/working/alphabet_results

In [ ]:
from IPython.display import Markdown, display
import os

for f in ["1_apostrophes.md", "2_conversion_audit.md", "3_tokenizers.md"]:
    p = f"/kaggle/working/alphabet_results/{f}"
    if os.path.exists(p):
        display(Markdown(open(p, encoding="utf-8").read()))

## 5. What do people actually type? (census on raw web text)
The held-out set was decoded from `val.bin`, which the tokenizer had already normalized, so the census in step 4 only shows which marks *survived* normalization. This step streams 3,000 documents of raw FineWeb-2 text and runs the census and conversion audit on them. About 2 minutes. Tokenizers are skipped: these documents come from the training region, so fertility here would flatter `uzbek-bpe-16k`.

In [ ]:
import json
from datasets import load_dataset

ds = load_dataset("HuggingFaceFW/fineweb-2", name="uzn_Latn", split="train", streaming=True)
n = 0
with open("/kaggle/working/raw_sample.jsonl", "w", encoding="utf-8") as f:
    for ex in ds:
        t = (ex.get("text") or "").strip()
        if t:
            f.write(json.dumps({"text": t}, ensure_ascii=False) + "\n")
            n += 1
        if n >= 3000:
            break
print(f"saved {n} raw documents")

In [ ]:
!python alphabet_study.py --heldout /kaggle/working/raw_sample.jsonl --out /kaggle/working/alphabet_results_raw --skip-tokenizers

from IPython.display import Markdown, display
display(Markdown(open("/kaggle/working/alphabet_results_raw/1_apostrophes.md", encoding="utf-8").read()))

## Next
1. **Download** `/kaggle/working/alphabet_results/` from the Output tab.
2. **Hand-label** `2_disagreements.csv`: fill the `verdict` column with `naive_wrong`, `careful_wrong`, `both_ok` or `unclear`. That turns "plain find-and-replace breaks things" into a measured error rate.
3. **Compare** the `raw` or `norm` fertility of `uzbek-bpe-16k` with 1.749 in the paper's Table 1. That also re-measures fertility on text the tokenizer never saw.
4. **Retrain** two tokenizers on the same sample (one normalized, one in the new alphabet) — the test of whether the tokenizer is still the cheap lever after the reform. See the docstring of `alphabet_study.py`.